# Assessment 4 — Fine-tuning an LLM with LoRA and QLoRA

**Task:** Fine-tune a small open-source LLM (**TinyLlama-1.1B-Chat**) for a domain-specific task — **medical question answering** — using two parameter-efficient techniques:

1. **LoRA** — Low-Rank Adaptation. Freezes the base model in fp16 and only trains small low-rank adapter matrices.
2. **QLoRA** — Quantized LoRA. Loads the base model in **4-bit** precision (NF4 quantization) and trains LoRA adapters on top. Drastically lower memory.

**Dataset:** [MedQuAD](https://www.kaggle.com/datasets/jpmiller/layoutlm) — ~47,000 medical Q&A pairs from NIH sources (we subsample to ~2,000 for Colab free tier).

**Hardware:** Google Colab free tier (T4 GPU, 16 GB VRAM). Total runtime ≈ 30–45 minutes.

---
**Before running:** Runtime → Change runtime type → **GPU (T4)**.

## 1. Install dependencies

In [ ]:
# Pin to known-good versions for reproducibility on Colab free tier
!pip install --upgrade torchao
!pip install trl torch --upgrade
!pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U \
    "transformers>=4.44" \
    "peft>=0.12" \
    "trl>=0.11" \
    "accelerate>=0.33" \
    "bitsandbytes>=0.43" \
    "datasets>=2.20" \
    kagglehub

## 2. Imports and environment check

In [ ]:
import os, gc, json, random
import numpy as np
import pandas as pd
import torch




from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "GPU not available — set Runtime → Change runtime type → GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Kaggle credentials and dataset download

Upload your `kaggle.json` (Kaggle → Settings → API → Create New API Token, legacy format).

In [ ]:
from google.colab import files
print("Please upload your kaggle.json:")
files.upload()

!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("jpmiller/layoutlm")
print("Dataset downloaded to:", dataset_path)
print("Files:", os.listdir(dataset_path))

## 4. Load and explore MedQuAD

In [ ]:
# Locate medquad.csv (case-insensitive)
csv_path = None
for f in os.listdir(dataset_path):
    if f.lower() == 'medquad.csv':
        csv_path = os.path.join(dataset_path, f)
        break
assert csv_path is not None, "medquad.csv not found"

df = pd.read_csv(csv_path)
print(f"Total rows: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

In [ ]:
# Basic cleaning: drop rows with missing question or answer, drop very short/long entries
df = df.dropna(subset=['question', 'answer']).copy()
df['question'] = df['question'].astype(str).str.strip()
df['answer']   = df['answer'].astype(str).str.strip()

# Filter: keep reasonable-length entries (helps fit max_seq_length=512)
df = df[df['question'].str.len().between(10, 300)]
df = df[df['answer'].str.len().between(20, 1500)]
print(f"After cleaning: {len(df):,} rows")

# Subsample to keep training fast on free tier
N_SAMPLES = 2000
df = df.sample(n=min(N_SAMPLES, len(df)), random_state=SEED).reset_index(drop=True)
print(f"Subsampled to: {len(df):,} rows")

# Quick look
print("\nSample question:")
print(df.iloc[0]['question'])
print("\nSample answer:")
print(df.iloc[0]['answer'][:400], '...')

## 5. Format examples as chat-style training samples

We use TinyLlama's chat template so the model learns to follow the same format used at inference time.

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

SYSTEM_PROMPT = (
    "You are a helpful medical assistant. Provide accurate, concise, evidence-based "
    "answers to medical questions. If unsure, say so."
)

def build_text(question, answer):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": question},
        {"role": "assistant", "content": answer},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

df['text'] = df.apply(lambda r: build_text(r['question'], r['answer']), axis=1)

# Train / eval split
df_eval  = df.sample(n=50, random_state=SEED)
df_train = df.drop(df_eval.index).reset_index(drop=True)
df_eval  = df_eval.reset_index(drop=True)
print(f"Train: {len(df_train)}, Eval: {len(df_eval)}")

train_ds = Dataset.from_pandas(df_train[['text']])
print("\nFormatted example (truncated):\n")
print(df_train.iloc[0]['text'][:600], '...')

## 6. Helper functions: sample generation and memory cleanup

In [ ]:
def generate_answer(model, tokenizer, question, max_new_tokens=200):
    """Generate an answer to a single medical question."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    ).strip()

def free_memory():
    """Release GPU memory between training runs."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Pick a few medical questions from the eval set for qualitative comparison later
EVAL_QUESTIONS = df_eval['question'].head(3).tolist()
print("Eval questions:")
for q in EVAL_QUESTIONS:
    print("  -", q[:120])

## 7. Baseline: generations from the *un-tuned* base model

So we have something to compare the fine-tuned versions against.

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto",
)

baseline_outputs = {}
for q in EVAL_QUESTIONS:
    ans = generate_answer(base_model, tokenizer, q)
    baseline_outputs[q] = ans
    print(f"Q: {q}")
    print(f"A (baseline): {ans}\n{'-'*80}")

# Free memory before LoRA training
del base_model
free_memory()

## 8. Part A — LoRA fine-tuning (fp16 base)

We load TinyLlama in **fp16** (no quantization), then attach LoRA adapters of rank 8 to the attention projections. Only the adapters are trained; the base weights stay frozen.

In [ ]:
# 8.1 Load base model in fp16
model_lora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto",
)
model_lora.config.use_cache = False  # required for gradient checkpointing
model_lora.gradient_checkpointing_enable()

# 8.2 LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# 8.3 SFT training configuration
sft_args_lora = SFTConfig(
    output_dir="./lora_out",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    max_length=512,
    packing=False,
    dataset_text_field="text",
    seed=SEED,
)

trainer_lora = SFTTrainer(
    model=model_lora,
    train_dataset=train_ds,
    args=sft_args_lora,
    peft_config=lora_config,
)

# Print trainable parameter count
trainer_lora.model.print_trainable_parameters()

In [ ]:
# 8.4 Train LoRA
trainer_lora.train()

# Save LoRA adapter
trainer_lora.model.save_pretrained("./lora_adapter")
print("LoRA adapter saved to ./lora_adapter")

In [ ]:
# 8.5 Generate samples with LoRA-tuned model
lora_outputs = {}
for q in EVAL_QUESTIONS:
    ans = generate_answer(trainer_lora.model, tokenizer, q)
    lora_outputs[q] = ans
    print(f"Q: {q}")
    print(f"A (LoRA): {ans}\n{'-'*80}")

# Free memory before QLoRA
del trainer_lora, model_lora
free_memory()

## 9. Part B — QLoRA fine-tuning (4-bit base)

Same model, but loaded in **4-bit NF4 quantization** with double quantization. Memory drops to roughly 1/4 of fp16, while final quality is typically comparable. Uses `BitsAndBytesConfig` + `prepare_model_for_kbit_training`.

In [ ]:
# 9.1 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_qlora = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model_qlora.config.use_cache = False
model_qlora = prepare_model_for_kbit_training(model_qlora)

# 9.2 LoRA configuration (same as before)
qlora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# 9.3 SFT training configuration
sft_args_qlora = SFTConfig(
    output_dir="./qlora_out",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    max_length=512,
    packing=False,
    dataset_text_field="text",
    seed=SEED,
)

trainer_qlora = SFTTrainer(
    model=model_qlora,
    train_dataset=train_ds,
    args=sft_args_qlora,
    peft_config=qlora_config,
)
trainer_qlora.model.print_trainable_parameters()

In [ ]:
# 9.4 Train QLoRA
trainer_qlora.train()

trainer_qlora.model.save_pretrained("./qlora_adapter")
print("QLoRA adapter saved to ./qlora_adapter")

In [ ]:
# 9.5 Generate samples with QLoRA-tuned model
qlora_outputs = {}
for q in EVAL_QUESTIONS:
    ans = generate_answer(trainer_qlora.model, tokenizer, q)
    qlora_outputs[q] = ans
    print(f"Q: {q}")
    print(f"A (QLoRA): {ans}\n{'-'*80}")

del trainer_qlora, model_qlora
free_memory()

## 10. Side-by-side comparison: Baseline vs LoRA vs QLoRA

For each evaluation question, show the answers produced by all three model variants. This is the primary qualitative deliverable for the PDF report.

In [ ]:
def truncate(s, n=350):
    s = s.replace('\n', ' ').strip()
    return s if len(s) <= n else s[:n] + '...'

print("=" * 100)
for i, q in enumerate(EVAL_QUESTIONS, 1):
    print(f"\n[Question {i}] {q}\n")
    print(f"  BASELINE: {truncate(baseline_outputs[q])}\n")
    print(f"  LoRA:     {truncate(lora_outputs[q])}\n")
    print(f"  QLoRA:    {truncate(qlora_outputs[q])}\n")
    print("-" * 100)

# Save comparison to a file (useful for the PDF submission)
comparison = {
    'questions': EVAL_QUESTIONS,
    'baseline':  baseline_outputs,
    'lora':      lora_outputs,
    'qlora':     qlora_outputs,
}
with open('comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2)

# Pretty markdown export for inclusion in the PDF
with open('comparison.md', 'w') as f:
    f.write("# LoRA vs QLoRA — Generation Comparison\n\n")
    for i, q in enumerate(EVAL_QUESTIONS, 1):
        f.write(f"## Question {i}\n\n**{q}**\n\n")
        f.write(f"**Baseline:** {baseline_outputs[q]}\n\n")
        f.write(f"**LoRA:** {lora_outputs[q]}\n\n")
        f.write(f"**QLoRA:** {qlora_outputs[q]}\n\n---\n\n")
print("Saved comparison.json and comparison.md")

## 11. Loading a saved adapter for inference

This is what you would do after training: load the base model + the LoRA adapter on top, and use it to answer new questions. Demonstrated here for the QLoRA adapter (the lighter-weight version).

In [ ]:
# Reload base in 4-bit and attach the QLoRA adapter
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
)
inference_model = PeftModel.from_pretrained(base, "./qlora_adapter")

new_question = "What are the early symptoms of type 2 diabetes?"
answer = generate_answer(inference_model, tokenizer, new_question, max_new_tokens=250)
print(f"Q: {new_question}\n\nA: {answer}")

del inference_model, base
free_memory()

## 12. Save outputs and download

The two adapter folders (`lora_adapter/`, `qlora_adapter/`) are tiny (~10 MB each) since they only contain the LoRA matrices, not the full model weights.

In [ ]:
# Zip the adapters for easy download
!zip -qr lora_adapter.zip lora_adapter
!zip -qr qlora_adapter.zip qlora_adapter
!ls -lh lora_adapter.zip qlora_adapter.zip comparison.json comparison.md

In [ ]:
# Download key artifacts
from google.colab import files
for f in ['lora_adapter.zip', 'qlora_adapter.zip', 'comparison.json', 'comparison.md']:
    files.download(f)

FileNotFoundError: Cannot find file: lora_adapter.zip

## 13. Discussion: LoRA vs QLoRA

| Aspect | LoRA | QLoRA |
|---|---|---|
| Base model precision | fp16 (16-bit) | NF4 (4-bit) + double quantization |
| Peak GPU memory (TinyLlama-1.1B) | ~5–6 GB | ~2–3 GB |
| Trainable parameters | Same (only adapters) | Same (only adapters) |
| Training speed | Slightly faster | Slightly slower (de-quant cost per step) |
| Final quality | Strong baseline | Comparable on most tasks |
| Largest model on a 16 GB T4 | ≈ 3B params | ≈ 7B params |

**Takeaway:** QLoRA's 4-bit quantization is what unlocks fine-tuning of larger models on consumer GPUs. On a small model like TinyLlama-1.1B, both methods comfortably fit, so the comparison is mainly pedagogical here. On a 7B model, only QLoRA would fit on a free T4.

